In [1]:
!pip install markdownify beautifulsoup4 requests

Programa previo para ver la cantidad de páginas a rastrear

In [2]:
import requests
import time
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse, parse_qs, urlencode

# --- CONFIGURACIÓN ---
BASE_URL = "https://www.upv.es/entidades/asic/"
PAUSA_RED = 0.05
# ---------------------

visitadas = {} # URL: Profundidad mínima encontrada
cola = [(BASE_URL.rstrip('/'), 0)] # (URL, Profundidad)
conteo = {
    "html_es": 0,
    "pdf_es": 0,
    "saltadas_idioma": 0,
    "duplicados_tecnicos": 0,
    "errores": 0
}
max_prof_detectada = 0

def normalizar_upv(url):
    u = urlparse(url.lower().split('#')[0].rstrip('/'))
    params = parse_qs(u.query)
    solo_idioma = {k: v for k, v in params.items() if k == 'p_idioma'}
    query_limpia = urlencode(solo_idioma, doseq=True)
    return f"{u.scheme}://{u.netloc}{u.path}{'?' + query_limpia if query_limpia else ''}"

def es_español(url):
    u = url.lower()
    if any(x in u for x in ["p_idioma=v", "/va/", "/en/", "p_idioma=i"]):
        return False
    return True

print(f"🔍 ESCANEO MODO LITE (Profundidad + Conteo) en: {BASE_URL}\n")

try:
    while cola:
        url_cruda, prof_actual = cola.pop(0)
        url_norm = normalizar_upv(url_cruda)

        # Si ya la visitamos a una profundidad menor o igual, la saltamos
        if url_norm in visitadas and visitadas[url_norm] <= prof_actual:
            conteo["duplicados_tecnicos"] += 1
            continue

        if not es_español(url_norm):
            conteo["saltadas_idioma"] += 1
            visitadas[url_norm] = prof_actual
            continue

        try:
            time.sleep(PAUSA_RED)
            # HEAD para ver tipo de archivo
            res = requests.head(url_norm, timeout=5, headers={'User-Agent': 'UPV-Scanner-Lite'}, allow_redirects=True)
            visitadas[url_norm] = prof_actual

            if prof_actual > max_prof_detectada:
                max_prof_detectada = prof_actual

            tipo = res.headers.get("Content-Type", "")

            if "text/html" in tipo:
                conteo["html_es"] += 1
                res_full = requests.get(url_norm, timeout=5)
                soup = BeautifulSoup(res_full.text, 'html.parser')

                print(f"\rHTMLs: {conteo['html_es']} | PDFs: {conteo['pdf_es']} | Profundidad Máx: {max_prof_detectada} | Escaneando...", end="")

                for link in soup.find_all('a', href=True):
                    hijo_url = urljoin(url_norm, link['href'])
                    hijo_norm = normalizar_upv(hijo_url)
                    if hijo_url.startswith(BASE_URL) and hijo_norm not in visitadas:
                        cola.append((hijo_url, prof_actual + 1))

            elif "application/pdf" in tipo:
                conteo["pdf_es"] += 1

        except Exception:
            conteo["errores"] += 1

except KeyboardInterrupt:
    print("\n\n⏹️ Escaneo detenido por el usuario.")

print("\n" + "="*45)
print(" 📊 RESULTADOS DEL ANÁLISIS ESTRUCTURAL")
print("="*45)
print(f"📍 Profundidad máxima alcanzada: {max_prof_detectada} niveles")
print(f"📄 Páginas HTML (Español):       {conteo['html_es']}")
print(f"📎 Archivos PDF (Español):       {conteo['pdf_es']}")
print(f"🚫 Saltadas (Otros idiomas):      {conteo['saltadas_idioma']}")
print(f"❌ Errores encontrados:           {conteo['errores']}")
print("-"*45)
print(f"💡 El bot necesitará procesar {conteo['html_es'] + conteo['pdf_es']} archivos.")
print("="*45)

🔍 ESCANEO MODO LITE (Profundidad + Conteo) en: https://www.upv.es/entidades/asic/

HTMLs: 239 | PDFs: 24 | Profundidad Máx: 5 | Escaneando...
 📊 RESULTADOS DEL ANÁLISIS ESTRUCTURAL
📍 Profundidad máxima alcanzada: 5 niveles
📄 Páginas HTML (Español):       239
📎 Archivos PDF (Español):       24
🚫 Saltadas (Otros idiomas):      200
❌ Errores encontrados:           0
---------------------------------------------
💡 El bot necesitará procesar 263 archivos.


Extractor de estructura

In [4]:
import requests
import csv
import time
import os
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse, parse_qs, urlencode
from google.colab import drive

# --- CONFIGURACIÓN ---
BASE_URL = "https://www.upv.es/estudios/grado/index-es.html"
LIMITE_SCAN = 1000 # Escanearemos las primeras 1000 para la muestra
PATH_DRIVE = '/content/drive/My Drive/UPV_Bot_Data'
# ---------------------

drive.mount('/content/drive')
os.makedirs(PATH_DRIVE, exist_ok=True)

cola = [(BASE_URL, 0)] # (URL, Profundidad)
visitadas = {} # URL: Profundidad
inventario = []

print(f"🕵️ Iniciando mapeo estructural de las primeras {LIMITE_SCAN} páginas...")

try:
    while cola and len(visitadas) < LIMITE_SCAN:
        url_actual, prof = cola.pop(0)
        u_parsed = urlparse(url_actual)

        # Normalización básica para el conteo
        url_norm = f"{u_parsed.scheme}://{u_parsed.netloc}{u_parsed.path}"

        if url_norm in visitadas: continue
        visitadas[url_norm] = prof

        try:
            # Usamos un timeout corto para no ralentizar el mapeo
            res = requests.get(url_actual, timeout=5, headers={'User-Agent': 'UPV-Structure-Mapper'})

            ctype = res.headers.get("Content-Type", "").lower()
            size = len(res.content)

            # Registrar datos de la página
            inventario.append({
                "URL": url_actual,
                "Subdominio": u_parsed.netloc,
                "Profundidad": prof,
                "Tipo": "HTML" if "text/html" in ctype else "Otro",
                "Tamaño_KB": round(size / 1024, 2)
            })

            print(f"\r🔍 Mapeando: {len(visitadas)}/{LIMITE_SCAN} | Prof: {prof} | {u_parsed.netloc}", end="")

            if "text/html" in ctype:
                soup = BeautifulSoup(res.text, 'html.parser')
                for link in soup.find_all('a', href=True):
                    hijo = urljoin(url_actual, link['href']).split('#')[0].rstrip('/')
                    # Solo seguimos enlaces internos de la UPV
                    if hijo.endswith('.upv.es') or 'upv.es' in urlparse(hijo).netloc:
                        if hijo not in visitadas:
                            cola.append((hijo, prof + 1))

        except Exception as e:
            continue

finally:
    # Guardar los resultados en un CSV para análisis
    csv_report = os.path.join(PATH_DRIVE, "mapeo_estructural_tit_upv.csv")
    with open(csv_report, "w", newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=["URL", "Subdominio", "Profundidad", "Tipo", "Tamaño_KB"])
        writer.writeheader()
        writer.writerows(inventario)

    print(f"\n\n✅ Mapeo finalizado. Informe guardado en: {csv_report}")

    # --- MINI REPORTE EN CONSOLA ---
    subdominios = [d['Subdominio'] for d in inventario]
    print("\n📊 RESUMEN DE LA ESTRUCTURA DETECTADA:")
    print(f"----------------------------------------")
    from collections import Counter
    for sub, count in Counter(subdominios).most_common(10):
        print(f"🔹 {sub}: {count} páginas")

Mounted at /content/drive
🕵️ Iniciando mapeo estructural de las primeras 1000 páginas...
🔍 Mapeando: 1000/1000 | Prof: 2 | www.upv.es

✅ Mapeo finalizado. Informe guardado en: /content/drive/My Drive/UPV_Bot_Data/mapeo_estructural_tit_upv.csv

📊 RESUMEN DE LA ESTRUCTURA DETECTADA:
----------------------------------------
🔹 www.upv.es: 737 páginas
🔹 innovacion.upv.es: 79 páginas
🔹 www.alumni.upv.es: 54 páginas
🔹 www.cfp.upv.es: 42 páginas
🔹 futuritat.webs.upv.es: 12 páginas
🔹 aplicat.upv.es: 7 páginas
🔹 intranet.upv.es: 4 páginas
🔹 automatricula.upv.es: 4 páginas
🔹 poliformat.upv.es: 3 páginas
🔹 poseidon.cfp.upv.es: 3 páginas


Programa que extrae los datos

In [ ]:
import os
import requests
import hashlib
import csv
import time
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse, parse_qs, urlencode
import markdownify
from google.colab import drive

# Montamos tu Drive
drive.mount('/content/drive')

# Definimos la ruta en tu Drive (se creará una carpeta llamada 'UPV_Bot_Data')
GUARDAR_EN_DRIVE = True
PATH_BASE_DRIVE = '/content/drive/My Drive/UPV_Bot_Data'

if not os.path.exists(PATH_BASE_DRIVE):
    os.makedirs(PATH_BASE_DRIVE)

In [ ]:
import os, requests, hashlib, time, re, markdownify, io, pickle
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse, parse_qs, urlunparse
from google.colab import drive

# --- 1. CONFIGURACIÓN ---
PATH_BASE_DRIVE = '/content/drive/My Drive/UPV_Bot_Data_Final'
MAX_PAGINAS = 50000
ARCHIVO_ESTADO = os.path.join(PATH_BASE_DRIVE, "estado_bot.pkl")

try:
    from PyPDF2 import PdfReader
except ImportError:
    os.system('pip install PyPDF2')
    from PyPDF2 import PdfReader

# --- 2. FILTROS Y REGLAS DE EXCLUSIÓN ---
DOMINIOS_PROHIBIDOS = [
    'riunet.upv.es', 'intranet.upv.es', 'polilupa.upv.es', 'correo.upv.es',
    'search.upv.es', 'sede.upv.es', 'alumni.upv.es', 'wiki.upv.es', 'apps.upv.es'
]

URLS_PROHIBIDAS = [
    'p_idioma=v', 'p_idioma=i', '/va/', '/en/', 'index-va', 'index-en',
    '/ficha-personal/', '/bitstream/', 'handle/10251',
    'podcast.upv.es', 'media.upv.es', 'tv.upv.es', '.mp4', '.mp3', '.zip',
    'wp-login.php', 'sharer.php', '/feed'
]

# --- 3. LÓGICA DE LIMPIEZA Y VALIDACIÓN ---

def es_valida(url):
    u = urlparse(url)
    dom = u.netloc.lower()
    # Solo dominio UPV y que no esté en listas negras
    if not (dom == "upv.es" or dom.endswith(".upv.es")): return False
    if any(d in dom for d in DOMINIOS_PROHIBIDOS): return False
    if any(p in url.lower() for p in URLS_PROHIBIDAS): return False
    return True

def normalizar_url(url):
    """Elimina fragmentos y parámetros de ruido para evitar duplicados."""
    url = url.split('#')[0] # Quitar anclas
    url = re.sub(r'index(-es|c|v|en)?\.html', 'index.html', url) # Estandarizar index
    u = urlparse(url)
    params_ruido = {'tl', 'p_sesion', 'p_vista', 'lang', 'jsessionid', 'p_idioma', 'ie', 'q'}
    query = parse_qs(u.query.lower())
    query_limpia = {k: v for k, v in query.items() if k not in params_ruido}
    from urllib.parse import urlencode
    nueva_query = urlencode(query_limpia, doseq=True)
    return urlunparse((u.scheme, u.netloc.lower(), u.path.lower(), u.params, nueva_query, '')).rstrip('/')

# --- 4. FUNCIONES DE EXTRACCIÓN PROFUNDA ---

def pdf_to_text(content):
    try:
        f = io.BytesIO(content)
        reader = PdfReader(f)
        return "\n".join([p.extract_text() for p in reader.pages if p.extract_text()])
    except: return ""

def extraer_guia_docente_profunda(url_asig):
    """Navega por las pestañas de la asignatura para extraer el 100% de la info."""
    try:
        res = requests.get(url_asig, timeout=10, headers={'User-Agent': 'Mozilla/5.0'})
        soup = BeautifulSoup(res.text, 'html.parser')
        datos_full = [f"# GUÍA DOCENTE COMPLETA: {url_asig}\n"]
        # Buscamos los enlaces de las pestañas laterales/superiores
        pestanas = soup.find_all('a', string=re.compile(r'Unidades|Evaluación|Resultados|Descripción|Bibliografía', re.I))
        for p in pestanas:
            u_p = urljoin(url_asig, p['href'])
            try:
                r_p = requests.get(u_p, timeout=5)
                s_p = BeautifulSoup(r_p.text, 'html.parser')
                cuerpo = s_p.find(id='cuerpo') or s_p.find('main') or s_p.body
                datos_full.append(f"## SECCIÓN: {p.get_text(strip=True)}\n" + markdownify.markdownify(str(cuerpo)))
            except: continue
        return "\n\n".join(datos_full)
    except: return ""

def obtener_silo(url):
    u = url.lower()
    if any(x in u for x in ['boupv', 'normativa', 'estatutos', 'presupuesto', 'reglamento']): return '01_Legislacion'
    if '/titulaciones/' in u or '/estudios/' in u: return '02_Academico'
    if '/entidades/' in u or '/contenidos/' in u: return '03_Entidades'
    if 'noticia' in u or 'ndp-app' in u: return '04_Noticias'
    return '05_General'

# --- 5. GESTIÓN DE ESTADO Y DRIVE ---

drive.mount('/content/drive')
os.makedirs(PATH_BASE_DRIVE, exist_ok=True)

if os.path.exists(ARCHIVO_ESTADO):
    print("🔄 Reanudando sesión y depurando cola...")
    with open(ARCHIVO_ESTADO, 'rb') as f:
        datos = pickle.load(f)
        visitadas = datos.get('visitadas', set())
        hashes_contenido = datos.get('hashes_contenido', set())
        cola_previa = datos.get('cola', [])
        # Aplicamos los filtros nuevos a la cola cargada
        cola = []
        vistas_en_cola = set()
        for uc, pr in cola_previa:
            un = normalizar_url(uc)
            if es_valida(un) and un not in visitadas and un not in vistas_en_cola:
                cola.append((un, pr))
                vistas_en_cola.add(un)
    print(f"🧹 Cola optimizada: de {len(cola_previa)} a {len(cola)} URLs.")
else:
    print("🆕 Iniciando extracción desde cero...")
    visitadas, hashes_contenido = set(), set()
    cola = [
        ("https://www.upv.es/index.html", 0),
        ("https://www.upv.es/organizacion/la-institucion/index.html", 0),
        ("https://www.upv.es/estudios/grado/index.html", 0),
        ("https://www.upv.es/estudios/master/index.html", 0),
        ("https://www.upv.es/pls/oalu/est_noticias.buscadornoticias", 0),
        ("https://www.upv.es/entidades/SG/infoweb/sg/info/513084normalc.html", 0)
    ]

def guardar_progreso():
    with open(ARCHIVO_ESTADO, 'wb') as f:
        pickle.dump({'visitadas': visitadas, 'cola': cola, 'hashes_contenido': hashes_contenido}, f)

# --- 6. BUCLE DE PROCESAMIENTO ---

print(f"🚀 Ejecutando rastreador sin simplificaciones...")

try:
    while cola and len(visitadas) < MAX_PAGINAS:
        url_cruda, prof = cola.pop(0)
        url_norm = normalizar_url(url_cruda)

        if url_norm in visitadas or not es_valida(url_norm):
            continue

        try:
            print(f"🔍 [{len(visitadas)}] Analizando: {url_norm}")
            res = requests.get(url_norm, timeout=12, headers={'User-Agent': 'Mozilla/5.0'})
            visitadas.add(url_norm)

            # A. Extracción de contenido según tipo
            if url_norm.lower().endswith('.pdf'):
                content = pdf_to_text(res.content)
            elif 'detAsignatura' in url_norm:
                content = extraer_guia_docente_profunda(url_norm)
            else:
                if "text/html" not in res.headers.get("Content-Type", "").lower(): continue
                soup = BeautifulSoup(res.text, 'html.parser')

                # Fusión de IFrames (Oracle/Planes de estudio)
                iframe = soup.find('iframe', id='marco')
                if iframe and iframe.get('src'):
                    try:
                        res_if = requests.get(urljoin(url_norm, iframe['src']), timeout=10)
                        soup_if = BeautifulSoup(res_if.text, 'html.parser')
                        iframe.replace_with(soup_if)
                    except: pass

                # Extraer nuevos enlaces antes de limpiar el HTML
                for a in soup.find_all('a', href=True):
                    lnk = urljoin(url_norm, a['href'])
                    lnk_n = normalizar_url(lnk)
                    if es_valida(lnk_n) and lnk_n not in visitadas:
                        cola.append((lnk_n, prof + 1))

                # Limpieza de basura visual
                for s in soup(['header', 'footer', 'nav', 'script', 'style', '.global-menu', '.bg-overlay']): s.decompose()
                main_c = soup.find(id='cuerpo') or soup.find('main') or soup.body
                content = markdownify.markdownify(str(main_c), heading_style="ATX")

            # B. Validación de contenido y duplicados
            if not content.strip(): continue
            h = hashlib.md5(content.encode('utf-8')).hexdigest()
            if h in hashes_contenido: continue
            hashes_contenido.add(h)

            # C. Guardado Físico
            silo = obtener_silo(url_norm)
            u_p = urlparse(url_norm)
            path_parts = [p for p in u_p.path.strip('/').split('/') if p]
            folder = path_parts[1] if len(path_parts) > 1 else (path_parts[0] if path_parts else "root")

            # Nombre de archivo único (evita colisiones)
            nombre_f = path_parts[-1] if path_parts else "index"
            if u_p.query:
                nombre_f += "_" + hashlib.md5(u_p.query.encode()).hexdigest()[:6]

            ruta_dir = os.path.join(PATH_BASE_DRIVE, silo, folder)
            os.makedirs(ruta_dir, exist_ok=True)

            with open(os.path.join(ruta_dir, f"{nombre_f[:100]}.md"), "w", encoding="utf-8") as f:
                f.write(f"--- INFO EXTRA ---\nURL: {url_norm}\nCAPTURADO: {time.ctime()}\n---\n\n" + content)

            if len(visitadas) % 50 == 0:
                guardar_progreso()
                print(f"💾 Checkpoint: {len(cola)} en cola.")

        except Exception: continue

finally:
    guardar_progreso()
    print(f"📊 Finalizado. Total archivos en Drive: {len(visitadas)}")